In [2]:
import pandas as pd
import numpy as np

def load_orderbook(csv_path: str) -> pd.DataFrame:
    """
    Load the LOB snapshot CSV, parse event timestamps, sort, and index by timestamp.
    """
    df = pd.read_csv(csv_path)
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    df = df.sort_values(['symbol', 'ts_event']).set_index('ts_event')
    return df

def compute_best_level_ofi(df: pd.DataFrame, level: int = 0) -> pd.Series:
    lvl = f"{level:02d}"
    bid_size = df[f'bid_sz_{lvl}']
    ask_size = df[f'ask_sz_{lvl}']

    d_bid = bid_size.diff().fillna(0)
    d_ask = ask_size.diff().fillna(0)

    bid_plus  = d_bid.clip(lower=0)
    bid_minus = (-d_bid).clip(lower=0)
    ask_plus  = d_ask.clip(lower=0)
    ask_minus = (-d_ask).clip(lower=0)

    ofi = (bid_plus - bid_minus) - (ask_plus - ask_minus)
    return ofi.rename(f'best_ofi_L{level}')

def compute_multi_level_ofi(df: pd.DataFrame, max_level: int = 2) -> pd.Series:
    ofis = [compute_best_level_ofi(df, lvl) for lvl in range(max_level + 1)]
    multi = pd.concat(ofis, axis=1).sum(axis=1)
    return multi.rename(f'multi_ofi_0to{max_level}')

def compute_integrated_ofi(ofi_series: pd.Series, window: str = '1min') -> pd.Series:
    # drop any duplicate timestamps so resampling and later concat won't explode
    ofi_series = ofi_series[~ofi_series.index.duplicated(keep='first')]
    integrated = ofi_series.resample(window).sum()
    return integrated.rename(f'integrated_ofi_{window}')

def compute_cross_asset_ofi(
    df_x: pd.DataFrame, df_y: pd.DataFrame, level: int = 0
) -> pd.DataFrame:
    ofi_x = compute_best_level_ofi(df_x, level)
    ofi_y = compute_best_level_ofi(df_y, level)
    cross = pd.concat([ofi_x, ofi_y], axis=1).dropna()
    cross.columns = [f'ofi_X_L{level}', f'ofi_Y_L{level}']
    return cross

def main():
    csv_path = 'first_25000_rows.csv'
    symbol_x = 'AAPL'
    symbol_y = 'MSFT'
    multi_max_level = 4
    integrate_window = '1min'

    lob = load_orderbook(csv_path)

    # filter per‐symbol and drop duplicate timestamps
    df_x = lob[lob['symbol'] == symbol_x].copy()
    df_x = df_x[~df_x.index.duplicated(keep='first')]
    df_y = lob[lob['symbol'] == symbol_y].copy()
    df_y = df_y[~df_y.index.duplicated(keep='first')]

    # 1) Best‐Level OFI for X
    best_ofi_x = compute_best_level_ofi(df_x, level=0)

    # 2) Multi‐Level OFI for X (levels 0..multi_max_level)
    multi_ofi_x = compute_multi_level_ofi(df_x, max_level=multi_max_level)

    # 3) Integrated OFI for X over fixed windows
    integrated_ofi_x = compute_integrated_ofi(best_ofi_x, window=integrate_window)

    # 4) Cross‐Asset OFI between X and Y at best level
    cross_ofi_xy = compute_cross_asset_ofi(df_x, df_y, level=0)

    # Combine into one DataFrame (no more duplicate‐index error)
    features = pd.concat(
        [best_ofi_x, multi_ofi_x, integrated_ofi_x, cross_ofi_xy],
        axis=1
    )

    features.to_csv('ofi_features_output.csv')
    print("Saved OFI features to ofi_features_output.csv")

if __name__ == "__main__":
    main()


Saved OFI features to ofi_features_output.csv
